# E1 single simulator pilot — guard-absent notebook path (2026-09-15)

Executes the motion side of `outputs/e1-pilot-execution-scope-2026-09-15.md` rev 3
against the **simulator only**. Code at `/Users/terrancehamilton/reachy-1-2-sim-e1` = `main` `f548fc6`.

* one setup traversal: `primitives.raise_to_side(reachy.r_arm)` (HOME → PRESENT via `PLACE_ROUTE` + `LIFT_TO_PRESENT`)
* one flight: `rig_motion.from_present(reachy.r_arm)` (= `fly_route(LOWER_TO_REST)`), only if a `go_flight` marker is written after every setup gate passed
* no retry, no recovery motion, no reset, no `place_object`, no second board

Each motion cell waits for the matching recorder to print `fly the route now`, sleeps the 3 s lead-in, then commands.
A `stop` marker in `control/` makes any pending motion cell do nothing.

In [ ]:
# Cell 1 — connect with literals (no env lookup for host/port); hygiene shown
import os, sys, time, json, pathlib, traceback
sys.path.insert(0, "/Users/terrancehamilton/reachy-1-2-sim-e1/src")
sys.path.insert(0, "/Users/terrancehamilton/reachy-1-2-sim-e1/scripts")
CTRL = pathlib.Path("/Users/terrancehamilton/reachy-1-2-sim-e1/docs/reviews/probes-2026-09-15-e1-tabletop-sim/control")
RECORD_ROOT = "/Users/terrancehamilton/reachy-1-2-sim-e1/docs/reviews/probes-2026-09-15-e1-tabletop-sim/e1_server_runs"
SCENE = "/Users/terrancehamilton/reachy-1-2-sim-e1/scenes/e1_boards/B4_pool_box_1_r2c3.yaml"
LEAD_IN_S = 3.0
print("REACHY env in this kernel:", {k: v for k, v in os.environ.items() if "REACHY" in k.upper()})
assert "REACHY_IP" not in os.environ and "REACHY_ENABLE_MOTION" not in os.environ, "shell hygiene violated"
from reachy_sdk import ReachySDK
HOST, PORT = "localhost", 50051
reachy = ReachySDK(host=HOST, sdk_port=PORT)
print(f"ReachySDK(host={HOST!r}, sdk_port={PORT}) connected at wall {time.time_ns()} mono {time.monotonic_ns()}")
print("python:", sys.executable)


In [ ]:
# Cell 2 — motion-client binding check: the recorder's own identity function, on THIS SDK object
import e1_identity
from reachy_ai.motion import rig_routes as R
def _read_sdk_joints():
    return {name: float(getattr(reachy.r_arm, name).present_position) for name in R.R_JOINTS}
ident = e1_identity.verify_simulator_identity(
    host=HOST, port=PORT, scene_path=SCENE, record_root=RECORD_ROOT,
    read_sdk_joints=_read_sdk_joints)
d = ident.as_dict()
print(json.dumps(d, indent=2, default=str))
BINDING_OK = bool(ident.ok)
(CTRL / ("binding_ok" if BINDING_OK else "binding_FAIL")).write_text(json.dumps(d, default=str))
print("BINDING_OK =", BINDING_OK)
print("present:", {k: round(v, 1) for k, v in _read_sdk_joints().items()})


In [ ]:
# Cell 3 — setup traversal: HOME -> PRESENT by primitives.raise_to_side (one attempt, no retry)
from reachy_ai.motion import primitives
def wait_for_recorder(logpath, timeout_s):
    t0 = time.monotonic()
    while time.monotonic() - t0 < timeout_s:
        if (CTRL / "stop").exists():
            return "stop"
        if logpath.exists() and "fly the route now" in logpath.read_text():
            return "ready"
        time.sleep(0.25)
    return "timeout"
SETUP = {"tool": "primitives.raise_to_side", "cell": 3}
status = wait_for_recorder(CTRL / "recorder_setup.log", 900) if BINDING_OK else "binding_failed"
SETUP["recorder_status"] = status
print("recorder status:", status)
if status == "ready":
    time.sleep(LEAD_IN_S)
    SETUP["t_start_mono_ns"] = time.monotonic_ns(); SETUP["t_start_wall_ns"] = time.time_ns()
    SETUP["start_pose"] = {k: round(v, 1) for k, v in _read_sdk_joints().items()}
    print("start pose:", SETUP["start_pose"])
    reachy.turn_on("r_arm")
    try:
        primitives.raise_to_side(reachy.r_arm)
        SETUP["outcome"] = "returned"
    except Exception as exc:
        SETUP["outcome"] = f"EXC {type(exc).__name__}: {exc}"
        traceback.print_exc()
    SETUP["t_end_mono_ns"] = time.monotonic_ns(); SETUP["t_end_wall_ns"] = time.time_ns()
    SETUP["elapsed_s"] = (SETUP["t_end_mono_ns"] - SETUP["t_start_mono_ns"]) / 1e9
    SETUP["end_pose"] = {k: round(v, 1) for k, v in _read_sdk_joints().items()}
    print("outcome:", SETUP["outcome"], "elapsed %.1f s" % SETUP["elapsed_s"])
    print("end pose:", SETUP["end_pose"])
else:
    SETUP["outcome"] = "not_attempted"
(CTRL / "setup_done").write_text(json.dumps(SETUP))


In [ ]:
# Cell 4 — the one flight: PRESENT -> REST_SHUT -> REST by rig_motion.from_present, only on go_flight
from reachy_ai.tasks import rig_motion
FLIGHT = {"tool": "rig_motion.from_present (= fly_route LOWER_TO_REST)", "cell": 4}
def wait_for_go(timeout_s):
    t0 = time.monotonic()
    while time.monotonic() - t0 < timeout_s:
        if (CTRL / "stop").exists():
            return "stop"
        if (CTRL / "go_flight").exists():
            return "go"
        time.sleep(0.5)
    return "timeout"
go = wait_for_go(1800) if (BINDING_OK and SETUP.get("outcome") == "returned") else "not_eligible"
FLIGHT["go"] = go
print("go:", go)
status = wait_for_recorder(CTRL / "recorder_flight.log", 900) if go == "go" else go
FLIGHT["recorder_status"] = status
print("recorder status:", status)
if status == "ready":
    time.sleep(LEAD_IN_S)
    FLIGHT["t_start_mono_ns"] = time.monotonic_ns(); FLIGHT["t_start_wall_ns"] = time.time_ns()
    FLIGHT["start_pose"] = {k: round(v, 1) for k, v in _read_sdk_joints().items()}
    print("start pose:", FLIGHT["start_pose"])
    phases = []
    try:
        reached = rig_motion.from_present(reachy.r_arm, on_phase=lambda *a: phases.append([time.monotonic_ns(), *map(str, a)]))
        FLIGHT["outcome"] = "returned"; FLIGHT["reached"] = reached
    except Exception as exc:
        FLIGHT["outcome"] = f"EXC {type(exc).__name__}: {exc}"
        traceback.print_exc()
    FLIGHT["phases"] = phases
    FLIGHT["t_end_mono_ns"] = time.monotonic_ns(); FLIGHT["t_end_wall_ns"] = time.time_ns()
    FLIGHT["elapsed_s"] = (FLIGHT["t_end_mono_ns"] - FLIGHT["t_start_mono_ns"]) / 1e9
    FLIGHT["end_pose"] = {k: round(v, 1) for k, v in _read_sdk_joints().items()}
    print("outcome:", FLIGHT["outcome"], "elapsed %.1f s" % FLIGHT["elapsed_s"])
    print("reached:", FLIGHT.get("reached")); print("phases:", phases)
    print("end pose:", FLIGHT["end_pose"])
else:
    FLIGHT["outcome"] = "not_attempted"
(CTRL / "flight_done").write_text(json.dumps(FLIGHT))


In [ ]:
# Cell 5 — final read-only state; no further motion, no turn_off
print("final pose:", {k: round(v, 1) for k, v in _read_sdk_joints().items()})
print("done at wall", time.time_ns())
